# 실습 2 — LoRA 파인튜닝: 요다(Yoda) 말투 학습

사전학습된 **Qwen2.5-1.5B** 모델을 요다 말투 데이터셋으로 파인튜닝하여,  
일반 영어 문장을 요다 스타일로 바꿔 말하는 모델을 만듭니다.

| 항목 | 내용 |
|---|---|
| **모델** | `Qwen2.5-1.5B-Instruct` (15억 파라미터) |
| **데이터셋** | `dvgodoy/yoda_sentences` (720개 문장 쌍) |
| **태스크** | 일반 영어 → 요다 말투 변환 |
| **방법** | LoRA Fine-tuning (전체의 ~1%만 학습) |

> **왜 요다 말투인가?**  
> 요다는 목적어를 앞으로 당기고 동사를 뒤로 미는 독특한 어순을 씁니다.  
> 베이스 모델은 이 말투를 전혀 모르기 때문에, 파인튜닝 **전후 차이가 극적으로** 보입니다.  
> 이것이 말투/스타일 파인튜닝의 핵심 원리입니다.

```
[파인튜닝 전]  "The Force is strong with you."
→ "The Force is indeed very strong with you."  (그냥 비슷하게 따라함)

[파인튜닝 후]  "The Force is strong with you."
→ "Strong with you, the Force is."             (요다 말투로 변환!)
```

## ⚙️ Colab GPU 설정

**설정 방법:**
1. 상단 메뉴 → **런타임** 클릭
2. **런타임 유형 변경** 클릭
3. 하드웨어 가속기 → **T4 GPU** 선택
4. **저장** 후 런타임 재시작

> ⚠️ GPU 없이 실행하면 학습 속도가 매우 느립니다. 반드시 설정 후 시작하세요.

## 📦 라이브러리 설치

| 라이브러리 | 역할 |
|---|---|
| `unsloth` | LLM을 빠르고 가볍게 파인튜닝하는 최적화 라이브러리 |
| `trl` | SFTTrainer 제공 — LLM 지도 파인튜닝 전용 Trainer |
| `peft` | LoRA 등 PEFT 기법을 적용하는 도구 |
| `datasets` | HuggingFace 데이터셋 로딩 |

In [ ]:
# ✅ STEP 1: 필수 라이브러리 설치
# ➤ Unsloth는 일반 transformers 대비 2배 빠른 속도와 60% 메모리 절약을 제공합니다.
!pip install bitsandbytes==0.48.0
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate  datasets
print("✅ 설치 완료")

In [ ]:
# ==========================================================
# ⚙️ STEP 0: 필수 환경 자동 세팅 (최초 1회 실행)
# 이 셀을 가장 먼저 실행해 주세요! 설치 완료 후 런타임이 자동 재시작됩니다.
# ==========================================================
import os
import time

print("📦 필수 라이브러리를 설치 중입니다... (약 2~3분 소요)")
print("설치 로그는 숨김 처리되며, 완료 후 화면이 살짝 깜빡이며 런타임이 재시작됩니다.")

# -q 옵션을 사용하여 길고 복잡한 설치 로그를 화면에서 숨깁니다.
!pip install -q vllm pyngrok openai peft accelerate bitsandbytes
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

print("✅ 설치가 완료되었습니다! 3초 뒤 런타임을 안전하게 자동 재시작합니다.")
time.sleep(3)

# Python 프로세스를 강제로 종료하여 코랩 런타임을 자동으로 재시작하는 명령어입니다.
# 사용자가 직접 상단 메뉴를 눌러 재시작할 필요가 없어집니다.
os.kill(os.getpid(), 9)

## 1단계. 모델 로딩

**Qwen2.5-1.5B-Instruct**는 15억 파라미터의 경량 생성 모델입니다.  
`FastLanguageModel`은 Unsloth가 제공하는 최적화된 로더로, 메모리를 훨씬 적게 사용합니다.

---
###load_in_4bit 설정

4비트 양자화를 활성화하면 모델 크기가 약 1/4로 줄어 Colab 무료 환경에서도 돌아갑니다.  
Colab T4 환경에서는 반드시 `True`로 설정해야 합니다.

---

In [ ]:
# ✅ STEP 2: 모델 및 토크나이저 로딩
# ➤ Unsloth의 FastLanguageModel은 일반 AutoModelForCausalLM보다 빠르고 메모리 효율적입니다.
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",  # 포인트: 1.5B 소형 모델 — Colab 무료 환경에 최적
    max_seq_length = 2048,   # 포인트: 한 번에 처리할 최대 토큰 수 (입력+출력 합계)
    load_in_4bit = True,      # ✏️ 4비트 양자화 활성화 여부 (True / False)
)

print("✅ 모델 로딩 완료: Qwen2.5-1.5B-Instruct")
print(f"전체 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

## 2단계. LoRA 어댑터 설정

LoRA는 전체 모델 가중치를 수정하지 않고, 각 가중치 행렬에 **작은 행렬 A와 B**를 추가로 붙여 그 부분만 학습합니다.

```
기존: output = W · input           (W는 고정, 수정 안 함)
LoRA: output = (W + A·B) · input   (A, B만 학습 — 전체의 ~1%)
```

---
###  LoRA 핵심 파라미터

| 파라미터 | 권장값 | 설명 |
|---|---|---|
| `r` | `16` | LoRA 행렬의 내부 차원(rank). 클수록 표현력↑, 메모리↑ |
| `lora_alpha` | `16` | LoRA 출력 스케일 조정값. 보통 r과 같게 설정 |
| `lora_dropout` | `0` | 드롭아웃 비율. 요다 720개처럼 작고 일관된 데이터에서는 0이 더 빠름 |

---

In [ ]:
# ✅ STEP 3: LoRA 어댑터 구성
# ➤ LoRA는 모델의 일부 레이어만 미세조정함으로써 빠르고 효율적인 파인튜닝이 가능합니다.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,             # ✏️ LoRA rank — 학습 파라미터 수 조절 (권장: 16)
    lora_alpha = 16,    # ✏️ 스케일 조정값 — 보통 r과 동일하게 설정
    lora_dropout = 0,  # ✏️ 드롭아웃 비율 (작고 일관된 데이터셋이므로 0 권장)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # 포인트: 어텐션 레이어 (Query/Key/Value/Output)
        "gate_proj", "up_proj", "down_proj"         # 포인트: FFN 레이어 (핵심 연산 담당)
        # 이 레이어들은 모델 연산의 핵심 부분입니다.
        # 큰 가중치를 가지므로 LoRA를 적용하면 적은 자원으로도 효과적으로 학습할 수 있습니다.
    ],
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # 포인트: 메모리 절약 기법 — 중간 계산값을 다시 계산해 저장 안 함
    random_state = 42,                       # 포인트: 재현성 확보 — 같은 시드면 같은 결과
)

# ✅ 실제로 학습되는 파라미터 비율 확인
# ➤ 전체의 1~2%만 학습됩니다. 이것이 LoRA의 핵심 장점입니다.
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"학습 파라미터: {trainable:,}")
print(f"전체 파라미터: {total:,}")
print(f"학습 비율    : {100 * trainable / total:.2f}%  ← 이 작은 비율만 학습합니다!")

## 3단계. 프롬프트 템플릿 & EOS 토큰

**Alpaca 스타일 프롬프트**는 모델이 일관된 형식으로 응답하도록 입출력 형식을 고정합니다.

```
### Instruction:  (태스크 설명 — "무엇을 해라")
### Input:        (사용자 입력 — 변환할 원문)
### Response:     (모델이 생성할 부분 — 요다 말투 번역)
```

---
### EOS 토큰

`EOS_TOKEN`은 모델이 응답을 마쳤다는 신호입니다.  
이것이 없으면 모델이 응답을 언제 끝내야 할지 몰라 계속 생성합니다.

---

In [ ]:
# ✅ STEP 4: Alpaca 스타일 프롬프트 포맷 정의
# ➤ 학습 데이터의 형식을 통일하기 위한 템플릿입니다.
#    instruction/input/response 세 칸을 고정 틀로 잡아 학습을 안정적으로 만듭니다.
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# ✅ EOS(문장 끝) 토큰 설정
# ➤ EOS 토큰을 붙여야 모델이 "여기까지가 정답"임을 명확히 알 수 있습니다.
EOS_TOKEN = tokenizer.eos_token  #  tokenizer에서 EOS 토큰을 가져오세요


print(f"EOS 토큰: {repr(EOS_TOKEN)}")
print("\n템플릿 예시:")
print(alpaca_prompt.format("Translate to Yoda-speak.", "The sky is blue.", "Blue, the sky is.") + EOS_TOKEN)

## 4단계. 데이터 포맷 변환 함수

요다 데이터셋 원본은 두 컬럼으로 구성됩니다.

```
sentence:    "The box was thrown beside the parked truck."  ← 일반 영어
translation: "Beside the parked truck, the box was thrown." ← 요다 말투
```

이를 Alpaca 프롬프트 형식으로 변환합니다.


In [ ]:
# ✅ STEP 5: 데이터 포맷 변환 함수 정의
# ➤ 원시 데이터의 두 컬럼(sentence/translation)을 하나의 텍스트로 변환합니다.
#    가공 로직을 한 곳에 모아 재사용 가능하게 만드는 것이 핵심입니다.
def format_instruction(example):
    return {
        "text": alpaca_prompt.format(
            "Translate the following English sentence into Yoda's speaking style.",                    # ✏️ 빈칸 6: instruction — 모델에게 무엇을 하라고 지시할까요?

            example["sentence"],      # 포인트: 원본 영어 문장 (input 자리)
            example["translation"]    # 포인트: 요다 말투 번역 (response 자리 — 정답)
        ) + EOS_TOKEN  # 포인트: 문장 끝에 EOS 토큰을 붙여 모델이 응답을 마무리하도록 유도
    }

# ✅ 변환 결과 미리 확인
test_ex = {
    "sentence": "The box was thrown beside the parked truck.",
    "translation": "Beside the parked truck, the box was thrown."
}
print(format_instruction(test_ex)["text"])

## 5단계. 데이터셋 로딩

**dvgodoy/yoda_sentences** 데이터셋은 720개의 영어-요다 말투 쌍으로 구성됩니다.

| 컬럼 | 예시 |
|---|---|
| `sentence` | "It's easy to tell the depth of a well." |
| `translation` | "Easy it is, to tell the depth of a well." |
| `translation_extra` | "Easy it is, to tell the depth of a well. Yes, hrrmmm." |

> 요다 말투의 핵심 패턴:  
> - 목적어/보어를 문장 앞으로 당김  
> - 동사를 뒤로 밀거나 `do/is`를 후치  
> - 간간이 `"Hrrmmm"`, `"Yes"` 같은 추임새 삽입

In [ ]:
# ✅ STEP 6: 데이터셋 로딩 및 포맷 변환
# ➤ HuggingFace에서 요다 문장 쌍 데이터를 불러오고 Alpaca 형식으로 변환합니다.
from datasets import load_dataset

dataset = load_dataset("dvgodoy/yoda_sentences")  # 포인트: 720개의 영어-요다 쌍 데이터셋

# ✅ 포맷 변환 적용
# ➤ 전체 데이터에 format_instruction 함수를 일괄 적용합니다.
train_data = dataset["train"].map(format_instruction, batched=False)  # 포인트: batched=False — 샘플 하나씩 변환

print(f"학습 데이터 수: {len(train_data)}개")
print(f"\n{'='*60}")
print("변환된 샘플 예시:")
print('='*60)
print(train_data[0]["text"])

## 6단계. 파인튜닝 전 베이스 모델 답변 확인

파인튜닝 **전**에 베이스 모델이 어떻게 답하는지 먼저 기록합니다.  
파인튜닝 후와 비교하기 위한 기준점(baseline)입니다.


In [ ]:
# ✅ STEP 7: 추론 함수 정의 (파인튜닝 전후 비교에 사용)
# ➤ 프롬프트를 구성하고 모델에서 응답을 생성하는 과정을 하나의 함수로 묶습니다.
def generate_response(model, tokenizer, sentence, max_new_tokens=60):
    # ✅ 추론용 프롬프트 구성
    # ➤ Response 칸은 비워두고 모델이 직접 채우도록 합니다.
    prompt = alpaca_prompt.format(
        "Translate the following English sentence into Yoda's speaking style.",
        sentence,  # ✏️ 변환할 영어 문장(sentence)을 넣으세요
        ""    # 포인트: Response 칸을 비워두면 모델이 이 자리를 채워 생성합니다
    )

    # ✅ 텍스트 → 토큰 변환 후 GPU로 이동
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)  # 포인트: .to(device) 필수 — GPU에서 추론

    with torch.no_grad():  # 포인트: 추론 시 gradient 계산 불필요 — 메모리 절약
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,  # 포인트: 생성할 최대 토큰 수 제한
            do_sample=False,                # 포인트: 확률 샘플링 없이 가장 확률 높은 토큰 선택 → 재현성 확보
            pad_token_id=tokenizer.eos_token_id,
        )

    # ✅ 입력 프롬프트 부분을 제거하고 생성된 부분만 디코딩
    # ➤ inputs["input_ids"].shape[1] 이후부터가 모델이 새로 생성한 토큰들입니다.
    generated = outputs[0][inputs["input_ids"].shape[1]:]  # 포인트: 슬라이싱으로 프롬프트 제거
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# ✅ 파인튜닝 전후 비교에 사용할 테스트 케이스
test_cases = [
    {"sentence": "The Force is strong with you.",
     "yoda": "Strong with you, the Force is."},
    {"sentence": "You must learn patience.",
     "yoda": "Patience, you must learn."},
    {"sentence": "Fear leads to anger.",
     "yoda": "Fear leads to anger."},
    {"sentence": "A great warrior you have become.",
     "yoda": "A great warrior you have become."},
    {"sentence": "There is no try.",
     "yoda": "No try, there is."},
]

print("="*65)
print("[BEFORE] 파인튜닝 전 베이스 모델 답변")
print("="*65)
for tc in test_cases:
    result = generate_response(model, tokenizer, tc["sentence"])
    is_yoda = any(w in result for w in ["you must", "is,", "do,", "Hrrmmm", "hrrrm"])
    mark = "🟡 요다 흉내?" if is_yoda else "❌ 일반 영어"
    print(f"원문  : {tc['sentence']}")
    print(f"정답  : {tc['yoda']}")
    print(f"출력  : {result[:80]}")
    print(f"판정  : {mark}")
    print("-"*65)

## 7단계. 학습 설정 및 SFTTrainer 구성

**SFTTrainer**는 LLM 생성 파인튜닝에 특화된 Trainer입니다.  
`dataset_text_field`만 지정하면 토크나이징, 배치 구성, 손실 계산을 모두 자동으로 처리합니다.

---
### dataset_text_field 지정

`SFTTrainer`에게 학습 데이터에서 **어떤 컬럼을 텍스트로 쓸지** 알려줘야 합니다.  
`format_instruction` 함수에서 어떤 키로 텍스트를 저장했는지 확인해보세요.

---

In [ ]:
# ✅ STEP 8: 학습 설정 및 SFTTrainer 구성
# ➤ 학습 방식(배치 크기, 학습률, 에폭 등)을 설정하고 Trainer를 초기화합니다.
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    per_device_train_batch_size = 4,     # 포인트: GPU 하나가 한 번에 처리할 샘플 수
    gradient_accumulation_steps = 4,     # 포인트: 4번 모아 큰 배치(4×4=16)처럼 학습 — 메모리 절약
    num_train_epochs = 3,                # 포인트: 전체 데이터를 3바퀴 반복 학습
    max_steps = 100,                     # 포인트: 빠른 테스트용 상한 (실제 학습 시 이 줄 제거)
    learning_rate = 2e-4,                # 포인트: 가중치 업데이트 크기 — 너무 크면 불안정, 너무 작으면 느림
    warmup_steps = 10,                   # 포인트: 초반 10 step 동안 학습률을 서서히 올려 안정적인 시작
    bf16 = is_bfloat16_supported(),      # 포인트: A100 등 최신 GPU에서 bfloat16 사용 (속도↑, 안정성↑)
    fp16 = not is_bfloat16_supported(),  # 포인트: T4 등 구형 GPU에서는 float16 사용
    logging_steps = 10,                  # 포인트: 10 step마다 loss 로그 출력
    optim = "adamw_8bit",               # 포인트: 8bit AdamW — 메모리 절약 + 속도 개선
    weight_decay = 0.01,                 # 포인트: 가중치에 패널티를 주어 과적합 방지
    lr_scheduler_type = "linear",        # 포인트: 학습률을 선형으로 감소 — 후반 과도한 업데이트 방지
    output_dir = "outputs",
    seed = 42,
    report_to = "none"                   # 포인트: wandb 등 외부 로깅 도구 비활성화
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    dataset_text_field = "sentence",  #텍스트가 담긴 컬럼 이름 — format_instruction에서 어떤 키를 썼나요?
    max_seq_length = 2048,
    dataset_num_proc = 2,        # 포인트: 전처리 병렬 처리 수
    packing = False,
    args = training_args,
)

print("✅ SFTTrainer 구성 완료")

## 8단계. 학습 시작

학습 로그에서 **`loss` 값이 점점 줄어드는지** 확인해보세요.  
loss가 낮아질수록 모델이 요다 말투 패턴을 학습하고 있다는 의미입니다.

> 예상 소요 시간: Colab T4 기준 약 **3~5분** (max_steps=100)

In [ ]:
# ✅ STEP 9: 학습 실행
# ➤ trainer.train() 한 줄로 순전파 → 손실 계산 → 역전파 → 파라미터 업데이트가 자동 반복됩니다.
trainer_stats = trainer.train()

print(f"\n✅ 학습 완료!")
print(f"총 학습 시간: {trainer_stats.metrics['train_runtime']:.1f}초")
print(f"최종 Loss   : {trainer_stats.metrics['train_loss']:.4f}")

## 9단계. 파인튜닝 후 비교

---
###FastLanguageModel 추론 모드 전환

파인튜닝이 끝난 모델을 추론에 최적화된 모드로 전환해야 합니다.  
Unsloth에서 추론 모드로 전환하는 메서드를 찾아보세요.

---

In [ ]:
# ✅ STEP 10: 파인튜닝 후 모델 답변 확인
# ➤ 파인튜닝 전과 동일한 테스트 케이스로 비교합니다.

# 모델을 추론(inference) 모드로 전환하세요
# 힌트: FastLanguageModel.for_inference(model)
# 추론 모드에서는 학습 관련 연산이 꺼지고 속도가 빨라집니다.
FastLanguageModel.for_inference(model)

print("="*65)
print("[AFTER] 파인튜닝 후 모델 답변")
print("="*65)
for tc in test_cases:
    result = generate_response(model, tokenizer, tc["sentence"])
    # 요다 말투 특징: 어순 변경, "you must", 동사 후치 등
    is_yoda_like = result.strip() != tc["sentence"].strip() and len(result) > 0
    mark = "✅ 변환됨!" if is_yoda_like else "❌ 변환 안 됨"
    print(f"원문  : {tc['sentence']}")
    print(f"정답  : {tc['yoda']}")
    print(f"출력  : {result[:80]}")
    print(f"판정  : {mark}")
    print("-"*65)

In [ ]:
# ✅ 파인튜닝 전후 나란히 비교
# ➤ 같은 입력에 대해 파인튜닝 전후 출력이 어떻게 달라졌는지 시각적으로 확인합니다.
print("\n" + "="*70)
print("📊 파인튜닝 전후 요약 비교")
print("="*70)

before_examples = [
    "Strong with you, the Force is...",    # 파인튜닝 전 예상 (일반 영어 반복)
    "You should learn to be patient.",
    "Anger comes from fear.",
]
after_examples = [
    "Strong with you, the Force is.",      # 파인튜닝 후 예상 (요다 말투)
    "Patience, you must learn.",
    "To anger, fear leads.",
]

print(f"{'입력':<35} {'파인튜닝 전':<35} {'파인튜닝 후'}")
print("-"*70)
for tc, bef, aft in zip(test_cases[:3], before_examples, after_examples):
    src = tc['sentence'][:30] + "..."
    print(f"{src:<35} {bef[:33]:<35} {aft}")
print()
print("⭐ 핵심: 파인튜닝 전에는 일반 영어로 비슷하게 반복하지만,")
print("         파인튜닝 후에는 어순이 바뀐 요다 말투로 변환됩니다!")

## 10단계. 새 문장으로 직접 테스트해보기

---
### ✏️ 빈칸 10 — 직접 문장을 넣어보세요

`my_sentence` 변수에 원하는 영어 문장을 입력하고,  
모델이 요다 말투로 어떻게 바꾸는지 확인해보세요!

---

In [ ]:
# ✅ STEP 11: 새 문장으로 직접 테스트
# ➤ 학습 데이터에 없던 새로운 문장을 넣어 스타일 학습 여부를 확인합니다.
#    모델이 단순히 외운 게 아니라 요다 말투 패턴 자체를 학습했다면 새 문장도 잘 변환합니다.

my_sentence = "You are the chosen one"  # ✏️ 빈칸 10: 요다 말투로 바꿔보고 싶은 영어 문장을 직접 입력해보세요!
                      # 예: "You are the chosen one." / "I will not give up." 등

result = generate_response(model, tokenizer, my_sentence)

print(f"원문   : {my_sentence}")
print(f"요다화 : {result}")
print()
print("직접 추가로 테스트해보세요:")
custom_tests = [
    "Knowledge is power.",
    "You cannot run from your destiny.",
    "The journey of a thousand miles begins with one step.",
]
for sent in custom_tests:
    out = generate_response(model, tokenizer, sent)
    print(f"  원문: {sent}")
    print(f"  요다: {out}")
    print()

## 11단계. LoRA 어댑터 저장

Full 모델 전체(수 GB)가 아닌, 학습한 **A, B 행렬만** 저장합니다.  
크기가 매우 작습니다(수 MB ~ 수십 MB).

---
### ✏️ 빈칸 11 — 저장 경로 지정

어댑터를 저장할 폴더 이름을 정해보세요. (예: `"./yoda-lora"`)

---

In [ ]:
# ✅ STEP 12: LoRA 어댑터 저장
# ➤ 전체 모델이 아닌 학습된 A, B 행렬(어댑터)만 저장합니다.
#    용량이 작고 나중에 베이스 모델 위에 다시 얹어 사용할 수 있습니다.
import os

SAVE_PATH = "./result"  # ✏️ 빈칸 11: 저장할 경로를 입력하세요 (예: "./yoda-lora")

model.save_pretrained(SAVE_PATH)    # 포인트: LoRA A, B 행렬만 저장 (전체 모델 아님)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ 저장 완료: {SAVE_PATH}")
print("\n저장된 파일 목록:")
for f in sorted(os.listdir(SAVE_PATH)):
    size = os.path.getsize(f"{SAVE_PATH}/{f}")
    print(f"  {f:45s}: {size/1024:.1f} KB")
print()
print("💡 adapter_model.safetensors가 실제 학습된 LoRA 가중치입니다.")
print("   전체 모델 대비 매우 작은 크기임을 확인하세요!")

In [ ]:
# ✅ (선택) HuggingFace Hub에 업로드
# ➤ Colab 세션이 종료되면 로컬 파일이 사라집니다.
#    HF Hub에 올려두면 언제든 다시 불러올 수 있고 팀원과 공유도 가능합니다.
from huggingface_hub import login

# 토큰 발급: https://huggingface.co/settings/tokens (write 권한 필요)
login(token="hf_자기토큰")

model.push_to_hub("zzmnxn/yoda-lora-qwen")
tokenizer.push_to_hub("zzmnxn/yoda-lora-qwen")

print("업로드 완료!")
print("→ https://huggingface.co/zzmnxn/yoda-lora-qwen")

#print("허깅페이스 업로드를 원하면 위 주석을 해제하고 토큰을 입력하세요.")

## ✅ 실습 2 완료!

### 전체 흐름 정리

```
모델 로딩 → LoRA 설정 → 프롬프트 포맷 → 데이터 변환 → 학습 전 확인
    → SFTTrainer 학습 → 학습 후 비교 → 어댑터 저장
```

### 핵심 포인트 3가지

| 포인트 | 설명 |
|---|---|
| **LoRA의 효율성** | 전체 파라미터의 ~1%만 학습해도 말투 변환이 가능 |
| **스타일 파인튜닝의 원리** | 베이스 모델이 모르는 말투를 쌍 데이터(입력-출력)로 주입 |
| **데이터 품질이 핵심** | 720개의 일관된 데이터가 수만 개의 불균일한 데이터보다 낫다 |

### 실제 응용: 챗봇 말투 학습

요다 실습에서 배운 원리를 그대로 응용하면 됩니다.

```python
# 챗봇 스타일 데이터 예시
{
    "sentence": "안녕하세요, 날씨가 좋네요.",    # 일반 말투
    "translation": "야 날씨 존나 좋다ㅋㅋ"       # 원하는 챗봇 말투
}
```

**데이터 만드는 법:** Claude/GPT에 원하는 말투를 설명하고 쌍 데이터 생성 요청 → LoRA 파인튜닝

In [ ]:
# =====================================================================
# 🚀 퀵 스타트: 허브에 올려둔 내 파인튜닝 모델 바로 써보기
# 앞선 학습 과정을 생략하고, 허브에서 모델을 다운받아 바로 추론합니다.
# =====================================================================

from unsloth import FastLanguageModel
import torch

# 1. 내가 허브에 올린 LoRA 어댑터 ID
# Unsloth는 이 ID만 주면 베이스 모델과 어댑터를 알아서 결합해 줍니다!
HF_MODEL_ID = "zzmnxn/yoda-lora-qwen"

print("📥 모델 다운로드 및 로딩 중... (약 1~2분 소요)")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = HF_MODEL_ID,
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,  # 메모리 절약을 위해 4bit 로드
)

# 2. 추론 전용 모드로 전환 (속도 2배 향상)
FastLanguageModel.for_inference(model)

# 3. 프롬프트 템플릿 (학습할 때와 동일한 템플릿 사용 필수)
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Translate the following English sentence into Yoda's speaking style.

### Input:
{}

### Response:
"""

# 4. 테스트할 문장 리스트 (딕셔너리 형태)
test_cases = [
    {"sentence": "The Force is strong with you.",
     "yoda": "Strong with you, the Force is."},
    {"sentence": "You must learn patience.",
     "yoda": "Patience, you must learn."},
    {"sentence": "Knowledge is the path to wisdom.",
     "yoda": "The path to wisdom, knowledge is."},
    {"sentence": "Fear leads to anger.",
     "yoda": "To anger, fear leads."},
    {"sentence": "Do or do not, there is no try.",
     "yoda": "Do or do not, there is no try."},
]

print("\n" + "="*60)
print("✨ 요다 모델 추론 결과 ✨")
print("="*60)

# 5. 반복문으로 추론 실행
for sent in test_cases:
    # 딕셔너리에서 키("sentence")로 입력 문장 가져오기
    input_text = sent["sentence"]

    # 템플릿에 문장 삽입
    prompt = alpaca_prompt.format(input_text)

    # 토크나이징 후 GPU로 전송
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # 텍스트 생성
    outputs = model.generate(**inputs, max_new_tokens=60, use_cache=True)

    # 결과 디코딩 및 파싱 (Response: 이후의 텍스트만 추출)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    final_answer = result.split("### Response:\n")[-1].strip()

    # 결과 출력
    print(f"👤 일반 영어: {input_text}")
    print(f"🎯 기대 정답: {sent['yoda']}")
    print(f"👽 모델 답변: {final_answer}")
    print("-" * 60)